In [29]:
import os
import pandas as pd
import numpy as np
import json
import shutil
from sklearn.model_selection import train_test_split
import torch
import torch.nn as nn
from torchvision import transforms
from PIL import Image
import xgboost as xgb
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

## Import the data

Ultrasound Image

In [30]:
us_data = '/media/shadab/Others/Liver_Fibrosis_US_processed'

In [31]:
classes = os.listdir(us_data)
print(f'Classes: {classes}')

Classes: ['F1', 'F2', 'F3', 'F4']


In [32]:
# print the number of files in each class
for cls in classes:
    cls_path = os.path.join(us_data, cls)
    num_files = len(os.listdir(cls_path))
    print(f'Number of files in class {cls}: {num_files}')

Number of files in class F1: 861
Number of files in class F2: 793
Number of files in class F3: 857
Number of files in class F4: 1698


Clinical Data

In [33]:
clinical_data = pd.read_csv('/media/shadab/Others/cirrhosis+patient+survival+prediction+dataset-1/clinical_data_processed.csv')

In [34]:
clinical_data.head()

,N_Days,Status,Drug,Age,Sex,Ascites,Hepatomegaly,Spiders,Edema,Bilirubin,...,Copper,Alk_Phos,SGOT,Tryglicerides,Platelets,Prothrombin,Stage,Ascites_missing,Hepatomegaly_missing,Spiders_missing
0,16.027556,3,1,124.941528,1,2,2,2,3,103.014440,...,52.794521,21.950609,52.539568,49.957522,51.842315,4.2,4.0,0,0,0
1,187.651451,1,1,116.094531,1,1,2,2,1,6.747292,...,18.037671,105.178334,41.257206,20.371681,64.155689,2.6,3.0,0,0,0
2,41.645562,3,1,168.079782,2,1,1,1,2,8.902527,...,71.195205,4.328053,33.212230,8.748673,36.351297,4.0,4.0,0,0,0
3,79.863273,3,1,109.587235,1,1,2,2,2,11.776173,...,21.445205,86.514845,16.831330,21.780531,49.061876,2.3,4.0,0,0,0
4,62.240429,2,3,46.122822,1,1,2,2,1,23.270758,...,48.364726,6.600513,41.086331,14.736283,30.393214,2.9,3.0,0,0,0


In [35]:
print(f"\nClass distribution:\n{clinical_data['Stage'].value_counts().sort_index()}")


Class distribution:
Stage
1.0     17
2.0     76
3.0    123
4.0     94
Name: count, dtype: int64


## Create Test Data

In [36]:
print("\n" + "="*80)
print("ANALYZING US IMAGE AVAILABILITY")
print("="*80)

# Count available US images per folder
us_counts = {}
for folder in ['F1', 'F2', 'F3', 'F4']:
    folder_path = os.path.join(us_data, folder)
    if os.path.exists(folder_path):
        num_files = len(os.listdir(folder_path))
        us_counts[folder] = num_files
        print(f"US images available in {folder}: {num_files}")
    else:
        us_counts[folder] = 0
        print(f"⚠️  {folder} not found")


ANALYZING US IMAGE AVAILABILITY
US images available in F1: 861
US images available in F2: 793
US images available in F3: 857
US images available in F4: 1698


In [37]:
print("\n" + "="*80)
print("CREATING HYBRID DATASET")
print("="*80)

# Create mapping from Stage values to folder names
stage_to_folder = {1.0: 'F1', 2.0: 'F2', 3.0: 'F3', 4.0: 'F4'}

# Build hybrid dataset by pairing clinical data with US images
hybrid_data = []
for idx, clinical_row in clinical_data.iterrows():
    stage = clinical_row['Stage']
    folder = stage_to_folder[stage]
    folder_path = os.path.join(us_data, folder)
    
    if os.path.exists(folder_path):
        us_images = os.listdir(folder_path)
        for us_image in us_images:
            us_image_path = os.path.join(folder_path, us_image)
            hybrid_data.append({
                'patient_idx': idx,
                'stage': stage,
                'folder': folder,
                'us_image': us_image,
                'us_image_path': us_image_path,
                'clinical_data': clinical_row
            })

hybrid_df = pd.DataFrame(hybrid_data)
print(f"Total paired samples created: {len(hybrid_df)}")
print(f"\nDistribution by stage:")
print(hybrid_df['folder'].value_counts().sort_index())


CREATING HYBRID DATASET
Total paired samples created: 339928

Distribution by stage:
folder
F1     14637
F2     60268
F3    105411
F4    159612
Name: count, dtype: int64
Total paired samples created: 339928

Distribution by stage:
folder
F1     14637
F2     60268
F3    105411
F4    159612
Name: count, dtype: int64


In [38]:
# Pick test cases - total ~60 cases based on availability
print("\n" + "="*80)
print("SELECTING TEST CASES (~60 total)")
print("="*80)

target_total = 60
stage_distribution = hybrid_df['stage'].value_counts().sort_index()

print(f"\nAvailable samples per stage:")
for stage, count in stage_distribution.items():
    print(f"  Stage {stage}: {count} samples")

# Calculate how many samples to pick from each stage
# Allocate proportionally based on availability
test_cases = []
total_selected = 0

for stage in sorted(hybrid_df['stage'].unique()):
    stage_data = hybrid_df[hybrid_df['stage'] == stage]
    available = len(stage_data)
    
    # Calculate proportional share of target_total
    proportion = available / len(hybrid_df)
    samples_for_stage = max(1, int(proportion * target_total))
    
    # Don't select more than available
    samples_for_stage = min(samples_for_stage, available)
    
    if samples_for_stage > 0:
        selected = stage_data.sample(n=samples_for_stage, random_state=42)
        test_cases.append(selected)
        total_selected += samples_for_stage
        print(f"  Stage {stage}: Selected {samples_for_stage} test cases (available: {available})")

test_hybrid = pd.concat(test_cases, ignore_index=True)

print(f"\n✅ Total test cases selected: {len(test_hybrid)}")
print("\nTest set class distribution:")
print(test_hybrid['folder'].value_counts().sort_index())


SELECTING TEST CASES (~60 total)

Available samples per stage:
  Stage 1.0: 14637 samples
  Stage 2.0: 60268 samples
  Stage 3.0: 105411 samples
  Stage 4.0: 159612 samples
  Stage 1.0: Selected 2 test cases (available: 14637)
  Stage 2.0: Selected 10 test cases (available: 60268)
  Stage 3.0: Selected 18 test cases (available: 105411)
  Stage 4.0: Selected 28 test cases (available: 159612)

✅ Total test cases selected: 58

Test set class distribution:
folder
F1     2
F2    10
F3    18
F4    28
Name: count, dtype: int64


In [39]:
# Create organized test dataset directory structure
print("\n" + "="*80)
print("ORGANIZING TEST SET")
print("="*80)

test_dataset_path = '../Hybrid_Test_Dataset'
os.makedirs(test_dataset_path, exist_ok=True)

# Create subdirectories for each stage
for stage in range(1, 5):
    stage_dir = os.path.join(test_dataset_path, f'Stage_{stage}')
    os.makedirs(stage_dir, exist_ok=True)
    os.makedirs(os.path.join(stage_dir, 'US_Images'), exist_ok=True)

print(f"Test dataset directory created at: {test_dataset_path}")

# Copy test set US images
print("\nCopying test set US images...")
copy_count = 0
for idx, row in test_hybrid.iterrows():
    stage_num = int(row['stage'])
    stage_dir = os.path.join(test_dataset_path, f'Stage_{stage_num}')
    us_dest = os.path.join(stage_dir, 'US_Images', row['us_image'])
    
    # Copy image if not already copied
    if not os.path.exists(us_dest):
        try:
            shutil.copy(row['us_image_path'], us_dest)
            copy_count += 1
        except Exception as e:
            print(f"  ❌ Error copying {row['us_image']}: {e}")

print(f"\n✅ {copy_count} US images organized successfully!")


ORGANIZING TEST SET
Test dataset directory created at: ../Hybrid_Test_Dataset

Copying test set US images...

✅ 58 US images organized successfully!


In [40]:
# Save clinical data for test set
print("\n" + "="*80)
print("SAVING TEST SET CLINICAL DATA")
print("="*80)

# Extract clinical data for test set
test_clinical_data = []
for idx, row in test_hybrid.iterrows():
    clinical_row = row['clinical_data'].to_dict()
    clinical_row['us_image'] = row['us_image']
    clinical_row['Stage'] = row['stage']
    clinical_row['folder'] = row['folder']
    test_clinical_data.append(clinical_row)

test_clinical_df = pd.DataFrame(test_clinical_data)

# Save as CSV
test_dataset_path = '../Hybrid_Test_Dataset'
os.makedirs(test_dataset_path, exist_ok=True)

test_clinical_path = os.path.join(test_dataset_path, 'test_clinical_data.csv')
test_clinical_df.to_csv(test_clinical_path, index=False)
print(f"✅ Clinical data saved to: {test_clinical_path}")

# Save as JSON with stage information
test_metadata = {
    'total_samples': len(test_hybrid),
    'stage_distribution': test_hybrid['folder'].value_counts().to_dict(),
    'samples': []
}

for idx, row in test_hybrid.iterrows():
    test_metadata['samples'].append({
        'patient_idx': int(row['patient_idx']),
        'Stage': float(row['stage']),
        'folder': row['folder'],
        'us_image': row['us_image'],
    })

metadata_path = os.path.join(test_dataset_path, 'test_metadata.json')
with open(metadata_path, 'w') as f:
    json.dump(test_metadata, f, indent=4)
print(f"✅ Metadata saved to: {metadata_path}")

print("\n" + "="*80)
print("TEST SET SUMMARY")
print("="*80)
print(f"Total test samples: {len(test_hybrid)}")
print("\nStage distribution:")
for stage in sorted(test_hybrid['stage'].unique()):
    count = len(test_hybrid[test_hybrid['stage'] == stage])
    folder = stage_to_folder[stage]
    print(f"  Stage {stage} ({folder}): {count} samples")


SAVING TEST SET CLINICAL DATA
✅ Clinical data saved to: ../Hybrid_Test_Dataset/test_clinical_data.csv
✅ Metadata saved to: ../Hybrid_Test_Dataset/test_metadata.json

TEST SET SUMMARY
Total test samples: 58

Stage distribution:
  Stage 1.0 (F1): 2 samples
  Stage 2.0 (F2): 10 samples
  Stage 3.0 (F3): 18 samples
  Stage 4.0 (F4): 28 samples


## Remove Test Samples from Original Dataset

Remove selected test images and clinical data from the original dataset to ensure they are not used in training

In [41]:
# Delete selected test US images from original processed directory
print("\n" + "="*80)
print("REMOVING TEST IMAGES FROM ORIGINAL DATASET")
print("="*80)

deleted_images = 0
failed_deletions = []

for idx, row in test_hybrid.iterrows():
    us_image_path = row['us_image_path']
    
    if os.path.exists(us_image_path):
        try:
            os.remove(us_image_path)
            deleted_images += 1
        except Exception as e:
            failed_deletions.append((us_image_path, str(e)))
            print(f"  ❌ Failed to delete {us_image_path}: {e}")
    else:
        print(f"  ⚠️  Image not found: {us_image_path}")

print(f"\n✅ Deleted {deleted_images} US images from original dataset")

if failed_deletions:
    print(f"\n⚠️  Failed to delete {len(failed_deletions)} images")
else:
    print("✅ All test images successfully removed from training dataset")

# Display remaining image counts per folder
print("\nRemaining US images per folder (for training):")
for folder in ['F1', 'F2', 'F3', 'F4']:
    folder_path = os.path.join(us_data, folder)
    if os.path.exists(folder_path):
        num_files = len(os.listdir(folder_path))
        print(f"  {folder}: {num_files} images")


REMOVING TEST IMAGES FROM ORIGINAL DATASET

✅ Deleted 58 US images from original dataset
✅ All test images successfully removed from training dataset

Remaining US images per folder (for training):
  F1: 859 images
  F2: 783 images
  F3: 839 images
  F4: 1670 images


In [42]:
# Remove selected test cases from clinical data and save updated training set
print("\n" + "="*80)
print("REMOVING TEST CASES FROM CLINICAL DATA")
print("="*80)

# Get unique patient indices that were selected for test set
test_patient_indices = test_hybrid['patient_idx'].unique()
print(f"Patient indices selected for testing: {sorted(test_patient_indices)}")

# Create training clinical data by removing test patients
training_clinical_data = clinical_data[~clinical_data.index.isin(test_patient_indices)].copy()

print(f"\nOriginal clinical data: {len(clinical_data)} patients")
print(f"Test set: {len(test_patient_indices)} patients")
print(f"Training set: {len(training_clinical_data)} patients")

# Display training set class distribution
print("\nTraining set class distribution:")
print(training_clinical_data['Stage'].value_counts().sort_index())

# Save the training clinical data (overwriting the original file)
training_clinical_path = '/media/shadab/Others/cirrhosis+patient+survival+prediction+dataset-1/clinical_data_processed.csv'
training_clinical_data.to_csv(training_clinical_path, index=False)
print(f"\n✅ Updated clinical data saved to: {training_clinical_path}")
print("✅ Test patients removed from training dataset")

print("\n" + "="*80)
print("DATASET SPLIT COMPLETE")
print("="*80)
print(f"Training samples: {len(training_clinical_data)} clinical records")
print(f"Test samples: {len(test_hybrid)} paired US+Clinical samples")
print("\nTest data location: {test_dataset_path}")
print("Training data: Original locations (with test samples removed)")


REMOVING TEST CASES FROM CLINICAL DATA
Patient indices selected for testing: [np.int64(10), np.int64(13), np.int64(15), np.int64(16), np.int64(20), np.int64(25), np.int64(26), np.int64(29), np.int64(33), np.int64(36), np.int64(43), np.int64(58), np.int64(64), np.int64(72), np.int64(78), np.int64(88), np.int64(89), np.int64(90), np.int64(91), np.int64(96), np.int64(101), np.int64(109), np.int64(113), np.int64(117), np.int64(119), np.int64(123), np.int64(124), np.int64(137), np.int64(139), np.int64(142), np.int64(143), np.int64(151), np.int64(177), np.int64(184), np.int64(195), np.int64(196), np.int64(206), np.int64(207), np.int64(222), np.int64(234), np.int64(243), np.int64(250), np.int64(259), np.int64(265), np.int64(266), np.int64(267), np.int64(278), np.int64(289), np.int64(294), np.int64(295), np.int64(301)]

Original clinical data: 310 patients
Test set: 51 patients
Training set: 259 patients

Training set class distribution:
Stage
1.0     16
2.0     67
3.0    107
4.0     69
Name: